# Dark Matter Halos from Substrate Memory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gpartin/LFMPublicExperiments/blob/main/LFMPublicExperiments/notebooks/LFM_Dark_Matter_Chi_Memory.ipynb)

## What This Notebook Demonstrates

In LFM, what astrophysics calls \u201cdark matter\u201d is really **$\chi$-memory**: the substrate remembers where mass used to be.

Using **GOV-03** (fast $\chi$-response with memory):

$$\chi^2 = \chi_0^2 - g\langle |\Psi|^2 \rangle_\tau$$

where $\langle \cdot \rangle_\tau$ is a time-averaged energy density with memory window $\tau$.

**Experiment**: Place a mass source, then move it. The $\chi$-well persists at the *old* location \u2014 this is a dark matter halo.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters
NX, NY = 64, 64
dx = 1.0
dt = 0.05
CHI_0 = 19.0
C = 1.0
G_COUPLING = 0.5    # chi-energy coupling strength
TAU = 25.0           # memory time constant
decay = np.exp(-dt / TAU)

X, Y = np.meshgrid(np.arange(NX) * dx, np.arange(NY) * dx, indexing='ij')

def lap2d(f):
    return (np.roll(f,1,0) + np.roll(f,-1,0)
            + np.roll(f,1,1) + np.roll(f,-1,1) - 4*f) / dx**2

def mass_source(cx, cy, amp=3.0, w=4.0):
    return amp * np.exp(-((X - cx)**2 + (Y - cy)**2) / (2 * w**2))

# Initialize energy memory (this is what drives chi via GOV-03)
energy_memory = np.zeros((NX, NY))

phases = [
    ("A: Mass at LEFT (x=16)", 500, 16.0, True),
    ("B: Mass at RIGHT (x=48)", 500, 48.0, True),
    ("C: NO mass (memory test)", 500, None, False),
]
snapshots = []

for label, steps, cx, active in phases:
    print(f"Running {label} ({steps} steps)...", end=" ")
    for step in range(steps):
        if active:
            # Mass source injects energy density
            src = mass_source(cx, 32.0)
            energy_memory = energy_memory * decay + src**2 * (1 - decay)
        else:
            # No source -- memory decays passively
            energy_memory = energy_memory * decay

    # GOV-03: algebraic chi response with memory
    chi_sq = CHI_0**2 - G_COUPLING * energy_memory
    chi = np.sqrt(np.maximum(chi_sq, 0.01))
    snapshots.append((label, chi.copy()))
    print(f"chi_min = {chi.min():.4f}")

# Memory test: does chi-well persist at OLD location after mass moved?
bg = snapshots[-1][1][0, 0]
old_loc = snapshots[-1][1][16, 32]
new_loc = snapshots[-1][1][48, 32]
memory_persists = old_loc < bg - 0.01
print(f"\nPhase C results (no mass anywhere):")
print(f"  chi at OLD mass location (16,32): {old_loc:.4f}")
print(f"  chi at NEW mass location (48,32): {new_loc:.4f}")
print(f"  chi at background (0,0):          {bg:.4f}")
print(f"  Memory persists at old location: {memory_persists}")
if memory_persists:
    print(f"\n  The chi-well PERSISTS where mass USED to be!")
    print(f"  This is what astrophysics calls a 'dark matter halo'.")
    print(f"  No dark matter particles needed -- just substrate memory.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (label, chi) in enumerate(snapshots):
    ax = axes[i]
    im = ax.imshow(chi.T, origin='lower', cmap='RdBu_r',
                   vmin=chi.min(), vmax=CHI_0 + 0.01,
                   extent=[0, NX, 0, NY])
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    plt.colorbar(im, ax=ax, label='chi')

    # Mark mass position
    if "LEFT" in label:
        ax.plot(16, 32, 'w*', markersize=15, label='Mass')
    elif "RIGHT" in label:
        ax.plot(48, 32, 'w*', markersize=15, label='Mass')
        ax.plot(16, 32, 'wx', markersize=12, label='Old position')
    else:
        ax.plot(16, 32, 'wx', markersize=12, label='Old position')
        ax.plot(48, 32, 'w+', markersize=12, label='Recent position')
    ax.legend(loc='upper right', fontsize=8)

plt.suptitle('Dark Matter = chi-Memory: Substrate Remembers Where Mass Was',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Cross-section plot
fig, ax = plt.subplots(1, 1, figsize=(12, 4))
for label, chi in snapshots:
    ax.plot(chi[:, 32], label=label, linewidth=1.5)
ax.axhline(CHI_0, color='gray', ls='--', alpha=0.5, label=f'chi_0 = {CHI_0}')
ax.axvline(16, color='green', ls=':', alpha=0.5, label='Old mass position')
ax.axvline(48, color='orange', ls=':', alpha=0.5, label='New mass position')
ax.set_xlabel('x')
ax.set_ylabel('chi')
ax.set_title('Cross-Section at y=32: chi-well persistence = dark matter halo')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Key Result

- Mass creates a $\chi$-well via energy density coupling
- When mass **moves away**, the $\chi$-well **persists** due to memory ($\tau = 25$)
- This persistent gravitational well is indistinguishable from a **dark matter halo**
- No dark matter particles needed \u2014 it\u2019s substrate memory